# Xây tokenizer từ số 0: BPE → SuperBPE

Language model không đọc được chữ. Nó chỉ nhận một dãy **số nguyên**, mỗi số là một "token".
Tokenizer là cái máy biến chữ thành dãy số (encode) và biến ngược lại (decode).

Notebook này xây tokenizer từng bước, mỗi bước sửa một vấn đề của bước trước. Cuối cùng ta sẽ có đúng thứ
`vitok.train_tokenizers` làm trên Kaggle, chỉ nhỏ hơn. Cứ chạy từng cell và nhìn output.

In [70]:
import collections
import unicodedata

text = """Học sinh ở Hà Nội đang học bài.
Các học sinh của trường đang học tiếng Anh.
Hà Nội là thủ đô của Việt Nam.
Tôi đang học ở Hà Nội, năm 2026.
Giáo viên của các học sinh rất vui.
Người dân Hà Nội đang chuẩn bị đón Tết.
Việt Nam có nhiều học sinh giỏi.
Các bạn học sinh đang đi học ở Hà Nội.
Thủ đô của Việt Nam là Hà Nội.
Học sinh của các trường ở Hà Nội đang nghỉ Tết."""
text2 = unicodedata.normalize('NFD', text)

print(len(text), "chars")
print(len(text2), "chars")

366 chars
459 chars


## Bước 1: chữ → số

Cách đơn giản nhất: mỗi ký tự là một số (Unicode code point).

In [71]:
print([ord(ch) for ch in "Hà Nội"])

[72, 224, 32, 78, 7897, 105]


Vấn đề: Unicode có ~150.000 ký tự, vocab sẽ quá to, và gặp ký tự lạ là bó tay.

Cách của GPT-2 (và project này): dùng **byte UTF-8**. Chỉ có 256 giá trị byte, mà văn bản nào cũng viết được bằng byte.
Đổi lại, chữ tiếng Việt có dấu tốn 2–3 byte:

In [72]:
for ch in "aàộđ":
    print(repr(ch), "->", list(ch.encode("utf-8")))

'a' -> [97]
'à' -> [195, 160]
'ộ' -> [225, 187, 153]
'đ' -> [196, 145]


Cùng một chữ còn có hai cách lưu. **NFC**: `ộ` là 1 ký tự. **NFD**: `o` + dấu nặng + dấu mũ, 3 ký tự.
Đây chính là biến NFC/NFD trong đề tài.

In [73]:
for form in ("NFC", "NFD"):
    s = unicodedata.normalize(form, "Nội")
    print(f"{form}: {len(s):2d} chars, {list(s.encode("utf-8"))}")

NFC:  3 chars, [78, 225, 187, 153, 105]
NFD:  5 chars, [78, 111, 204, 163, 204, 130, 105]


In [75]:
ids = list(text.encode("utf-8"))
print(len(text), "char ->", len(ids), "byte")
print(ids[:10])

366 char -> 490 byte
[72, 225, 187, 141, 99, 32, 115, 105, 110, 104]


In [76]:
ids = list(text2.encode("utf-8"))
print(len(text2), "char ->", len(ids), "byte")
print(ids[:10])

459 char -> 562 byte
[72, 111, 204, 163, 99, 32, 115, 105, 110, 104]


Giờ văn bản là một dãy số trong khoảng 0–255. Model train được trên dãy này, nhưng nó **quá dài**:
context có hạn, và mỗi token tốn một bước tính. Ta muốn nén dãy lại.

## Bước 2: ý tưởng BPE, gộp cặp hay gặp nhất

Đếm xem cặp số liền nhau nào xuất hiện nhiều nhất:

In [23]:
def render(b: bytes) -> str:
    """In chuỗi byte cho người đọc; byte UTF-8 chưa đủ thành ký tự hiện dạng \\xNN."""
    return "'" + b.decode("utf-8", errors="backslashreplace").replace("\n", "\\n") + "'"

def get_stats(ids, counts=None, weight=1):
    """Đếm các cặp liền kề. weight dùng ở bước 5."""
    counts = {} if counts is None else counts
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + weight
    return counts

stats = get_stats(ids)
top = sorted(stats.items(), key=lambda kv: -kv[1])[:5]
for (a, b), c in top:
    print((a, b), "xuất hiện", c, "lần:", render(bytes([a, b])))

(225, 187) xuất hiện 38 lần: '\xe1\xbb'
(99, 32) xuất hiện 14 lần: 'c '
(104, 225) xuất hiện 11 lần: 'h\xe1'
(187, 141) xuất hiện 10 lần: '\xbb\x8d'
(141, 99) xuất hiện 10 lần: '\x8dc'


Cặp đứng đầu `(225, 187)` là 2 byte đầu chung của `ọ`, `ộ`, `ở`, ... Tiếng Việt có dấu nên các cặp byte kiểu này rất nhiều.
Ý tưởng: **tạo token mới số 256 thay cho cặp đó**, rồi thay mọi chỗ xuất hiện.

In [24]:
def merge(ids, pair, idx):
    """Thay mọi lần xuất hiện của `pair` trong ids bằng token mới `idx`."""
    out, i = [], 0
    while i < len(ids):
        if i + 1 < len(ids) and (ids[i], ids[i + 1]) == pair:
            out.append(idx)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out

print(merge([5, 6, 6, 7, 9, 1], (6, 7), 99))  # ví dụ nhỏ

pair = top[0][0]
ids2 = merge(ids, pair, 256)
print(len(ids), "->", len(ids2))

[5, 6, 99, 9, 1]
490 -> 452


## Bước 3: lặp lại, và đó là training

Lặp: đếm cặp → gộp cặp nhiều nhất thành token mới 257, 258, ... Mỗi lần ta ghi lại:
- `merges[(a, b)] = idx`: luật gộp, **theo thứ tự học**.
- `vocab[idx] = bytes`: token đó là chuỗi byte nào (để decode và để in ra xem).

Token mới có thể gộp với token cũ, nên dần dần byte → chữ có dấu → âm tiết.

In [25]:
def train_naive(ids, num_merges, verbose=True):
    merges, vocab = {}, {i: bytes([i]) for i in range(256)}
    for k in range(num_merges):
        stats = get_stats(ids)
        pair = max(stats, key=stats.get)
        idx = 256 + k
        ids = merge(ids, pair, idx)
        merges[pair] = idx
        vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
        if verbose:
            print(f"{idx}: {render(vocab[pair[0]]):>12} + {render(vocab[pair[1]]):<12} -> {render(vocab[idx]):<16} ({stats[pair]} lần)")
    return merges, vocab, ids

merges, vocab, compressed = train_naive(list(text.encode("utf-8")), 40)
print(f"\n{len(text.encode())} byte -> {len(compressed)} token, nén {len(text.encode()) / len(compressed):.2f}x")

256:       '\xe1' + '\xbb'       -> '\xe1\xbb'       (38 lần)
257:          'c' + ' '          -> 'c '             (14 lần)
258:          'h' + '\xe1\xbb'   -> 'h\xe1\xbb'      (11 lần)
259:       '\x8d' + 'c '         -> '\x8dc '         (10 lần)
260:       '\xc3' + '\xa0'       -> 'à'              (10 lần)
261:          ' ' + 'N'          -> ' N'             (10 lần)
262:          ' ' + '\xc4'       -> ' \xc4'          (10 lần)
263:      ' \xc4' + '\x91'       -> ' đ'             (10 lần)
264:          'n' + 'g'          -> 'ng'             (10 lần)
265:          '.' + '\n'         -> '.\n'            (9 lần)
266:          ' ' + 'c'          -> ' c'             (9 lần)
267:          'n' + 'h'          -> 'nh'             (8 lần)
268:  'h\xe1\xbb' + '\x8dc '     -> 'học '           (8 lần)
269:          'H' + 'à'          -> 'Hà'             (7 lần)
270:         'Hà' + ' N'         -> 'Hà N'           (7 lần)
271:       'Hà N' + '\xe1\xbb'   -> 'Hà N\xe1\xbb'   (7 lần)
272: 'Hà N\xe1\

Đó là toàn bộ thuật toán train BPE. Không có gradient, không có neural net: chỉ đếm và gộp.

## Bước 4: encode và decode

**decode** dễ: nối các chuỗi byte lại. **encode** văn bản mới: bắt đầu từ byte, áp các luật gộp
**theo đúng thứ tự đã học** (luật học trước được ưu tiên, vì luật sau có thể dựa trên token của luật trước).

In [26]:
def decode(tokens, vocab):
    return b"".join(vocab[t] for t in tokens).decode("utf-8", errors="replace")

def encode_bytes(b: bytes, merges):
    ids = list(b)
    while len(ids) >= 2:
        stats = get_stats(ids)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))  # luật có thứ hạng thấp nhất
        if pair not in merges:
            break  # không còn cặp nào gộp được
        ids = merge(ids, pair, merges[pair])
    return ids

s = "Học sinh ở Hà Nội."
tokens = encode_bytes(s.encode("utf-8"), merges)
print(tokens)
print(" ".join(render(vocab[t]) for t in tokens))
assert decode(tokens, vocab) == s

[72, 256, 259, 289, 287, 46]
'H' '\xe1\xbb' '\x8dc ' 'sinh ' 'ở Hà Nội' '.'


## Bước 5: vấn đề, và cách GPT-2 sửa

Nhìn lại merge `257` ở bước 3: `'c '`, tức đuôi của "học" dính với dấu cách. Train nhiều merge hơn và liệt kê
các token có dấu cách ở giữa hoặc cuối. Vài cái là cụm có nghĩa (`'Hà Nội'`), nhưng nhiều cái là **mảnh lai**
(`'Hà N'`, `'hủ đ\xc3'`): phí chỗ trong vocab, và khó cho model học.

In [27]:
merges_naive, vocab_naive, _ = train_naive(list(text.encode("utf-8")), 60, verbose=False)
lai = [render(b) for i, b in vocab_naive.items() if i >= 256 and b" " in b[1:]]
print(len(lai), "token có dấu cách không nằm ở đầu:")
print(" ".join(lai))

21 token có dấu cách không nằm ở đầu:
'c ' '\x8dc ' 'học ' 'Hà N' 'Hà N\xe1\xbb' 'Hà Nộ' 'Hà Nội' ' Hà Nội' ' học ' 'ở Hà Nội' 'ác ' 'sinh ' ' đang học ' ' của ' 'Việt N' 'Việt Na' 'Việt Nam' 'Học ' 'i.\nCác ' 'hủ đ\xc3' 'hủ đô'


Cách sửa của GPT-2: **cắt văn bản thành mảnh bằng regex trước** (mỗi mảnh là một từ, kèm dấu cách đứng trước),
rồi chỉ cho gộp **bên trong** mỗi mảnh. Đây là regex stage 1 của project (có thêm `\p{M}` để dấu NFD không bị tách khỏi chữ).
Python `re` không hiểu `\p{L}`, nên ta mượn bộ cắt của thư viện `tokenizers`, đúng bộ mà `vitok` dùng.

In [28]:
from tokenizers import Regex, pre_tokenizers
from vitok.tokenizer_spec import STAGE1_REGEX, STAGE2_REGEX

def split_chunks(text: str, regex: str) -> list[str]:
    """HF train đọc file theo từng dòng, nên mảnh không bao giờ vượt qua xuống dòng."""
    split = pre_tokenizers.Split(pattern=Regex(regex), behavior="isolated", invert=False)
    return [p for line in text.splitlines(keepends=True) for p, _ in split.pre_tokenize_str(line)]

print(STAGE1_REGEX)
print(split_chunks("Tôi đang học ở Hà Nội, năm 2026.", STAGE1_REGEX))

[^\r\n\p{L}\p{N}]?[\p{Lu}\p{Lt}\p{Lm}\p{Lo}\p{M}]*[\p{Ll}\p{Lm}\p{Lo}\p{M}]+|[^\r\n\p{L}\p{N}]?[\p{Lu}\p{Lt}\p{Lm}\p{Lo}\p{M}]+[\p{Ll}\p{Lm}\p{Lo}\p{M}]*|\p{N}{1,3}| ?[^\s\p{L}\p{N}]+[\r\n/]*|\s*[\r\n]+|\s+(?!\S)|\s+
['Tôi', ' đang', ' học', ' ở', ' Hà', ' Nội', ',', ' năm', ' ', '202', '6', '.']


Một mẹo quan trọng: mảnh `' học'` xuất hiện rất nhiều lần, nhưng mảnh giống nhau thì gộp giống nhau.
Nên ta **chỉ lưu mỗi mảnh khác nhau một lần, kèm số lần xuất hiện**, và đếm cặp có nhân trọng số.
Trên 500MB văn bản, hàng trăm triệu mảnh chỉ còn vài trăm nghìn mảnh khác nhau. Đây là lý do train BPE nhanh.

In [29]:
chunks = collections.Counter(split_chunks(text, STAGE1_REGEX))
print(sum(chunks.values()), "mảnh, nhưng chỉ", len(chunks), "mảnh khác nhau")
print(chunks.most_common(8))

99 mảnh, nhưng chỉ 46 mảnh khác nhau
[('.\n', 9), (' học', 8), (' Nội', 7), (' sinh', 6), (' Hà', 6), (' đang', 6), (' của', 5), (' ở', 4)]


Viết lại hàm train cho dạng "mảnh + số đếm". Thêm hai thứ để dùng lại ở bước 7:
tham số `merges, vocab` để **train tiếp** từ một tokenizer có sẵn, và `max_words` để giới hạn số từ trong một token.

In [30]:
def train(chunks: collections.Counter, num_merges, merges=None, vocab=None, max_words=None, verbose=True):
    merges = dict(merges or {})
    vocab = dict(vocab or {i: bytes([i]) for i in range(256)})
    # mỗi mảnh khác nhau: (dãy token hiện tại, số lần xuất hiện)
    words = [(encode_bytes(c.encode("utf-8"), merges), n) for c, n in chunks.items()]
    banned, done = set(), 0
    while done < num_merges:
        stats = {}
        for ids, n in words:
            get_stats(ids, stats, weight=n)
        for p in banned:
            stats.pop(p, None)
        if not stats:
            break
        pair = max(stats, key=stats.get)
        new = vocab[pair[0]] + vocab[pair[1]]
        if max_words and (b": " in new or len([w for w in new.split(b" ") if w]) > max_words):
            banned.add(pair)  # luật của fork SuperBPE: token tối đa 4 từ, không chứa ": "
            continue
        idx = len(vocab)
        merges[pair], vocab[idx] = idx, new
        words = [(merge(ids, pair, idx), n) for ids, n in words]
        done += 1
        if verbose:
            print(f"{idx}: {render(vocab[pair[0]]):>14} + {render(vocab[pair[1]]):<14} -> {render(new):<22} ({stats[pair]} lần)")
    return merges, vocab

def encode(text: str, merges, regex: str):
    return [t for c in split_chunks(text, regex) for t in encode_bytes(c.encode("utf-8"), merges)]

## Bước 6: đây chính là BPE của project

Train 100 merge với regex stage 1. Để ý: byte → chữ có dấu → âm tiết, và **không token nào vượt dấu cách**.

Bản thật trong `vitok.train_tokenizers` giống hệt về ý tưởng, chỉ khác cỡ:
- vocab 16.000 = 256 byte + **15.744 merge**;
- train trên **500MB** (`tok_train.txt`). 500MB chỉ để *học luật gộp*;
- xong thì dùng tokenizer đó encode **10GB** văn bản khác (`shards/`) để *train language model* ở notebook 02.

(Trong file `merges.txt` của HuggingFace, byte được in thành ký tự lạ như `Ġ` = dấu cách, `á»` = 2 byte đầu của `ộ`.
Đó chỉ là cách in byte ra chữ, bản chất vẫn là những gì ta đang làm.)

In [31]:
N_MERGES = 100
bpe_merges, bpe_vocab = train(chunks, N_MERGES)

256:         '\xe1' + '\xbb'         -> '\xe1\xbb'             (38 lần)
257:            'h' + '\xe1\xbb'     -> 'h\xe1\xbb'            (11 lần)
258:         '\x8d' + 'c'            -> '\x8dc'                (10 lần)
259:         '\xc3' + '\xa0'         -> 'à'                    (10 lần)
260:            ' ' + 'N'            -> ' N'                   (10 lần)
261:            ' ' + '\xc4'         -> ' \xc4'                (10 lần)
262:        ' \xc4' + '\x91'         -> ' đ'                   (10 lần)
263:            'n' + 'g'            -> 'ng'                   (10 lần)
264:            '.' + '\n'           -> '.\n'                  (9 lần)
265:            ' ' + 'c'            -> ' c'                   (9 lần)
266:            'n' + 'h'            -> 'nh'                   (8 lần)
267:            ' ' + 'h\xe1\xbb'    -> ' h\xe1\xbb'           (8 lần)
268:   ' h\xe1\xbb' + '\x8dc'        -> ' học'                 (8 lần)
269:            'H' + 'à'            -> 'Hà'                   (7 lần

In [32]:
s = "Học sinh của các trường ở Hà Nội."
tokens = encode(s, bpe_merges, STAGE1_REGEX)
print("BPE:", len(tokens), "token:", " ".join(render(bpe_vocab[t]) for t in tokens))
assert decode(tokens, bpe_vocab) == s

BPE: 9 token: 'Học' ' sinh' ' của' ' các' ' trường' ' ở' ' Hà' ' Nội' '.'


## Bước 7: SuperBPE

Với tiếng Việt, mỗi mảnh stage 1 là **một âm tiết**. Nên `học sinh`, `Hà Nội`, `của các` luôn tốn ≥ 2 token,
dù chúng đi cùng nhau rất thường xuyên. SuperBPE hỏi: sao không cho gộp qua dấu cách?

Nhưng bước 5 vừa cho thấy: gộp qua dấu cách **ngay từ đầu** sinh ra token lai. Cách của SuperBPE là làm 2 giai đoạn:

1. **Stage 1**: BPE thường (có tách dấu cách), dừng sớm ở 80% số merge (project dùng 90%: 14.144 merge).
   Lúc này vocab đã có đủ âm tiết hoàn chỉnh.
2. **Stage 2**: đổi sang regex **không tách dấu cách**, rồi train tiếp phần merge còn lại. Vì đơn vị đã là âm tiết,
   cặp hay gặp nhất bây giờ là `' Hà' + ' Nội'`, không phải mảnh lai.

Regex stage 2 chỉ còn cắt ở số, chuỗi ≥2 dấu câu, và dấu cách cuối mảnh:

In [33]:
print(STAGE2_REGEX)
print(split_chunks("Tôi đang học ở Hà Nội, năm 2026.", STAGE2_REGEX))

\p{N}{1,3}| ?[^\s\p{L}\p{M}\p{N}]{2,}[\r\n/]*| +(?!\S)
['Tôi đang học ở Hà Nội, năm ', '202', '6', '.']


In [34]:
TRANSITION = 0.8
n_inherit = round(TRANSITION * N_MERGES)

# stage 1: lấy n_inherit merge đầu của BPE (chúng giống hệt việc train BPE rồi dừng sớm)
stage1_merges = dict(list(bpe_merges.items())[:n_inherit])
stage1_vocab = {i: b for i, b in bpe_vocab.items() if i < 256 + n_inherit}

# stage 2: cắt lại bằng regex mới, train tiếp từ tokenizer stage 1
chunks2 = collections.Counter(split_chunks(text, STAGE2_REGEX))
super_merges, super_vocab = train(chunks2, N_MERGES - n_inherit,
                                  merges=stage1_merges, vocab=stage1_vocab, max_words=4)

336:          ' Hà' + ' Nội'         -> ' Hà Nội'              (6 lần)
337:           ' ở' + ' Hà Nội'      -> ' ở Hà Nội'            (4 lần)
338:         ' học' + ' sinh'        -> ' học sinh'            (4 lần)
339:        ' đang' + ' học'         -> ' đang học'            (3 lần)
340:          'Học' + ' sinh'        -> 'Học sinh'             (2 lần)
341:          ' đô' + ' của'         -> ' đô của'              (2 lần)
342:      ' đô của' + ' Việt'        -> ' đô của Việt'         (2 lần)
343: ' đô của Việt' + ' Nam'         -> ' đô của Việt Nam'     (2 lần)
344:         ' của' + ' các'         -> ' của các'             (2 lần)
345:    ' học sinh' + ' '            -> ' học sinh '           (2 lần)
346:            'i' + '.\n'          -> 'i.\n'                 (2 lần)
347:    ' đang học' + ' bài'         -> ' đang học bài'        (1 lần)
348: ' đang học bài' + '.\n'          -> ' đang học bài.\n'     (1 lần)
349:          'Các' + ' học sinh'    -> 'Các học sinh'         (1 lần)
350: 

Cả hai tokenizer có **cùng cỡ vocab** (256 + 100). So sánh:

In [35]:
for name, m, v, regex in [("BPE", bpe_merges, bpe_vocab, STAGE1_REGEX),
                          ("SuperBPE", super_merges, super_vocab, STAGE2_REGEX)]:
    tokens = encode(s, m, regex)
    assert decode(tokens, v) == s
    print(f"{name:8s} {len(tokens)} token:", " ".join(render(v[t]) for t in tokens))

BPE      9 token: 'Học' ' sinh' ' của' ' các' ' trường' ' ở' ' Hà' ' Nội' '.'
SuperBPE 5 token: 'Học sinh' ' của các' ' trường' ' ở Hà Nội' '.'


Cùng số token trong vocab, nhưng SuperBPE nén câu còn ít token hơn. Câu hỏi của đề tài:
**model nhỏ học tốt hơn hay tệ hơn** khi mỗi token chứa nhiều chữ hơn (và khi dấu được tách ra kiểu NFD)?

## Bước 8: vì sao train thật bị treo trên Kaggle

Nhớ mẹo ở bước 5: BPE nhanh vì mảnh **ngắn và lặp lại nhiều**, nên chỉ phải xử lý mỗi mảnh khác nhau một lần.
Stage 2 phá mẹo đó: mảnh là cả cụm câu, gần như không bao giờ lặp lại nguyên văn.

In [36]:
for name, regex in [("stage 1", STAGE1_REGEX), ("stage 2", STAGE2_REGEX)]:
    c = collections.Counter(split_chunks(text, regex))
    total = sum(c.values())
    print(f"{name}: {total:3d} mảnh, {len(c):3d} khác nhau, mỗi mảnh lặp {total / len(c):.1f} lần, "
          f"dài trung bình {sum(len(k.encode()) for k in c) / len(c):5.1f} byte")

stage 1:  99 mảnh,  46 khác nhau, mỗi mảnh lặp 2.2 lần, dài trung bình   5.0 byte
stage 2:  13 mảnh,  13 khác nhau, mỗi mảnh lặp 1.0 lần, dài trung bình  37.7 byte


Trên 500MB văn bản thật (ước lượng từ 500 văn bản mẫu): stage 1 mỗi mảnh dài ~6 byte và lặp ~34 lần;
stage 2 dài ~174 byte và gần như không lặp. Trainer phải giữ gần như nguyên 500MB trong RAM dưới dạng mảnh riêng lẻ,
và mỗi lần gộp phải quét lại chúng. Hàm `train` ở trên cũng chịu đúng vấn đề đó, chỉ là dữ liệu ở đây quá nhỏ để thấy.

## Đối chiếu với code của project

| Notebook này | `vitok.train_tokenizers` trên Kaggle |
|---|---|
| `text` (10 câu) | `tok_train.txt` (500MB), chuẩn hoá NFC hoặc NFD trước |
| `split_chunks` + `STAGE1_REGEX` | `pre_tokenizers.Split(STAGE1_REGEX)` + `ByteLevel` |
| `train(chunks, 100)` | stage 1: `BpeTrainer(vocab_size=16000)` → điều kiện `bpe-nfc` / `bpe-nfd` |
| 80 merge đầu | 14.144 merge đầu, ghi vào `super-*/merges.txt` |
| `train(chunks2, ..., merges=stage1_merges, max_words=4)` | stage 2: fork SuperBPE thấy `merges.txt`, train tiếp đến 16.000 → `super-nfc` / `super-nfd` |
| `encode` / `decode` | `tokenizer.json`, dùng ở notebook 02 để encode 10GB pretrain |

## Tự thử

- Đổi `text = unicodedata.normalize("NFD", text)` ở cell đầu rồi chạy lại: các merge đầu gộp nguyên âm với dấu (`'o' + '\xcc'` → ... → `'ọ'`), vì mỗi dấu NFD là 2 byte `\xcc..`.
- Đổi `TRANSITION = 0.3`: stage 1 dừng quá sớm, âm tiết chưa hoàn chỉnh, xem stage 2 gộp ra token gì.
- Thêm câu vào `text` và xem superword nào xuất hiện.